# 07 - Python Toolbox Diagnostics

This notebook covers toolbox scripts, in-cluster diagnostics, memory usage, and GPU correlation.

Your current local profile keeps `pythonToolbox.enabled: false` by default to reduce RAM usage.


In [ ]:
import os
import sys
from pathlib import Path

expected = "/usr/local/bin/python3.11"
print(f"Python executable: {sys.executable}")
print(f"Python version   : {sys.version.split()[0]}")
if Path(sys.executable).resolve().as_posix() != expected:
    print(f"WARNING: This notebook is designed for {expected}.")
    print("Switch the notebook kernel to Python 3.11 before continuing.")
else:
    print("Kernel check passed.")


In [ ]:
import importlib
import subprocess
import sys

packages = ["requests", "pandas", "matplotlib", "pyyaml", "langsmith", "tabulate"]
missing = []
for pkg in packages:
    module_name = "yaml" if pkg == "pyyaml" else pkg
    try:
        importlib.import_module(module_name)
    except Exception:
        missing.append(pkg)

if missing:
    print("Installing missing packages:", ", ".join(missing))
    subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet", *missing])
else:
    print("All common packages are already installed.")


In [ ]:
import json
import os
import shlex
import subprocess
from pathlib import Path


def run(cmd: str, check: bool = True):
    print(f"$ {cmd}")
    proc = subprocess.run(cmd, shell=True, text=True, capture_output=True)
    if proc.stdout:
        print(proc.stdout.rstrip())
    if proc.stderr:
        print(proc.stderr.rstrip())
    if check and proc.returncode != 0:
        raise RuntimeError(f"Command failed ({proc.returncode}): {cmd}")
    return proc


def run_json(cmd: str):
    proc = run(cmd)
    return json.loads(proc.stdout)


PROJECT_ROOT = Path("/media/waqasm86/External1/Project-Nvidia-Office/Project-Llamatelemetry/langchain-kubernetes-jupyterlab/llm-observability-stack")
NAMESPACE = "llm-observability"
print(f"Project root: {PROJECT_ROOT}")


## Check Current Toolbox Status


In [ ]:
import yaml

values = yaml.safe_load((PROJECT_ROOT / "values.local-k3s.yaml").read_text())
print("pythonToolbox.enabled:", values.get("pythonToolbox", {}).get("enabled"))
run(f"kubectl get deploy -n {NAMESPACE} python-toolbox", check=False)
run(f"kubectl get pods -n {NAMESPACE} -l app.kubernetes.io/name=python-toolbox", check=False)


## Run Example Scripts If Toolbox Pod Exists


In [ ]:
pod_cmd = f"kubectl get pod -n {NAMESPACE} -l app.kubernetes.io/name=python-toolbox -o jsonpath='{{.items[0].metadata.name}}'"
proc = run(pod_cmd, check=False)
pod_name = proc.stdout.strip().strip("'") if proc.returncode == 0 else ""

if pod_name:
    print("Toolbox pod:", pod_name)
    run(f"kubectl exec -n {NAMESPACE} {pod_name} -- python /opt/examples/langsmith_healthcheck.py", check=False)
    run(f"kubectl exec -n {NAMESPACE} {pod_name} -- python /opt/examples/ollama_smoke.py", check=False)
else:
    print("No toolbox pod found. Enable it when needed:")
    print("helm upgrade --install llm-observability-stack . -n llm-observability -f values.local-k3s.yaml --set pythonToolbox.enabled=true")


## Pod Memory and CPU Profiling


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

top_proc = run(f"kubectl top pods -n {NAMESPACE} --no-headers", check=False)
lines = [line for line in top_proc.stdout.splitlines() if line.strip()]

rows = []
for line in lines:
    parts = line.split()
    if len(parts) >= 3:
        rows.append({"pod": parts[0], "cpu": parts[1], "memory": parts[2]})

top_df = pd.DataFrame(rows)
display(top_df)

if not top_df.empty:
    plot_df = top_df.copy()
    plot_df["memory_mi"] = (
        plot_df["memory"]
        .str.replace("Mi", "", regex=False)
        .str.replace("Gi", "000", regex=False)
        .astype(float)
    )
    ax = plot_df.sort_values("memory_mi", ascending=False).plot(
        x="pod", y="memory_mi", kind="bar", figsize=(11, 4), title="Pod Memory Usage (approx Mi)"
    )
    ax.set_ylabel("Memory (Mi approx)")
    plt.xticks(rotation=45, ha="right")
    plt.show()


## Correlate Inference With GPU Activity


In [ ]:
import requests
import time

print("GPU before inference:")
run("nvidia-smi --query-gpu=utilization.gpu,memory.used,memory.total --format=csv", check=False)

payload = {
    "model": "gemma3-1b-it-gguf-local",
    "stream": False,
    "messages": [{"role": "user", "content": "Describe how Kubernetes cronjobs work in 3 lines."}],
}

try:
    started = time.perf_counter()
    resp = requests.post("http://localhost:8000/ollama/api/chat", json=payload, timeout=240)
    elapsed = (time.perf_counter() - started) * 1000
    print("inference status:", resp.status_code, "latency_ms:", round(elapsed, 2))
except Exception as exc:
    print(f"Inference request skipped/failed: {exc}")

print("GPU after inference:")
run("nvidia-smi --query-gpu=utilization.gpu,memory.used,memory.total --format=csv", check=False)


## Search for Long-Running Python Processes in Stack Pods


In [ ]:
pods_json = run_json(f"kubectl get pods -n {NAMESPACE} -o json")
pod_names = [item["metadata"]["name"] for item in pods_json.get("items", [])]

for pod in pod_names:
    cmd = (
        f"kubectl exec -n {NAMESPACE} {pod} -- "
        "sh -lc \"ps -eo pid,etime,%cpu,%mem,comm,args | grep -E 'python|uvicorn|jupyter' | grep -v grep || true\""
    )
    print("=" * 120)
    print(pod)
    run(cmd, check=False)

## Done
You now have repeatable diagnostics for toolbox scripts, memory usage, and GPU-associated inference checks.
